# Feature Engineering

In [ ]:
import polars as pl
from pathlib import Path
import tomllib
import os

In [ ]:
config_file_path = Path("src/config.toml")

with config_file_path.open("rb") as config_file:
    config = tomllib.load(config_file)

COMPANIES = config['companies']

In [ ]:
company = next(company for company in COMPANIES)

In [ ]:
lf = pl.scan_parquet(f'ETL/data/silver/parquet/{company}_share_prices_cleaned.parquet')

In [ ]:
lf = lf.with_columns(
    # Log Return
    (pl.col('Adj. Close') / pl.col('Adj. Close').shift(1)).log().alias(f'Log Return')
)

for window in [5, 10, 20]:
    lf = lf.with_columns(
        # Rolling Log Returns
        (pl.col('Adj. Close') / pl.col('Adj. Close').shift(window)).log().alias(f'Log Return {window}d'),

        # Volatility
        pl.col('Log Return').rolling_std(window).log1p().alias(f'Volatility {window}d'),

        # Moving Averages
        pl.col("Adj. Close").rolling_mean(5).alias(f"Moving Average {window}d"),

        # Momentum Pct. Change
        ((pl.col("Adj. Close") / pl.col("Adj. Close").shift(window)) - 1).alias(f"Momentum Pct. {window}d"),

        # Log Volume Ratio
        ((pl.col("Volume") / pl.col("Volume").rolling_mean(window))).log().alias(f"Log Volume Ratio {window}d"),
    )

    lf = lf.with_columns(
        # Log MA Ratio
        (pl.col("Adj. Close") / pl.col(f"Moving Average {window}d")).log().alias(f"Log MA Ratio {window}d")
    )

lf = lf.with_columns(
    # Intraday Returns
    ((pl.col('Close') / pl.col('Open')) - 1).alias('Intraday Pct. Return'),

    # Ranges
    (pl.col('High') - pl.col('Low')).alias('Range'),
    ((pl.col('High') - pl.col('Low')) / pl.col('Close')).alias('Range Pct.'),

    # Close Position
    (((pl.col('Close') - pl.col('Low')) / (pl.col('High') - pl.col('Low'))) - 0.5).alias('Close Position'),

    # Log Volume Change
    (pl.col("Volume") / pl.col("Volume").shift(1)).log().tanh().alias("Log Volume Change"),

    # Log Market Cap
    (pl.col("Adj. Close") * pl.col("Shares Outstanding")).log().alias("Log Market Cap"),

    # Dilution / Issuance
    (pl.col("Shares Outstanding") / pl.col("Shares Outstanding").shift(1) - 1).alias('Delta Pct. Dilution / Issuance'),

    # Volume Return Interaction
    (pl.col("Log Return") * pl.col("Log Volume Ratio 5d")).tanh().alias("Interaction Return Volume 5d"),
    (pl.col("Log Return") * pl.col("Log Volume Ratio 10d")).tanh().alias("Interaction Return Volume 10d"),
    (pl.col("Log Return") * pl.col("Log Volume Ratio 20d")).tanh().alias("Interaction Return Volume 20d"),

    # Volume Volatility Interaction
    (pl.col("Volatility 5d") * pl.col("Log Volume Ratio 5d")).tanh().alias("Interaction Volatility Volume 5d"),
    (pl.col("Volatility 10d") * pl.col("Log Volume Ratio 10d")).tanh().alias("Interaction Volatility Volume 10d"),
    (pl.col("Volatility 20d") * pl.col("Log Volume Ratio 20d")).tanh().alias("Interaction Volatility Volume 20d"),

    # Momentum Volatility Interaction
    (pl.col("Momentum Pct. 5d") * pl.col("Volatility 5d")).tanh().alias("Interaction Momentum Volatility 5d"),
    (pl.col("Momentum Pct. 10d") * pl.col("Volatility 10d")).tanh().alias("Interaction Momentum Volume 10d"),
    (pl.col("Momentum Pct. 20d") * pl.col("Volatility 20d")).tanh().alias("Interaction Momentum Volume 20d"),

    # Target Engineering
    (pl.col("Adj. Close").shift(-1) / pl.col("Adj. Close") - 1).alias("Target Return Metric"),
    ((pl.col("Adj. Close").shift(-1) / pl.col("Adj. Close") - 1) > 0).alias("Target Return Class")
)

In [ ]:
with pl.Config(tbl_rows=-1):
    display(lf.head(25).collect())

In [ ]:
lf = lf.drop([
    "Open", 
    "High",
    "Low",
    "Close",
    "Adj. Close",
    "Moving Average 5d",
    "Moving Average 10d",
    "Moving Average 20d",
    "Volume",
    "Range",
    "Shares Outstanding"
])

lf = lf.drop_nulls()

In [ ]:
with pl.Config(tbl_rows=-1):
    display(lf.head(21).collect())

In [ ]:
SILVER_CSV_DIR = os.path.join('ETL', 'data', 'silver', 'csv')
os.makedirs(SILVER_CSV_DIR, exist_ok=True)
filepath = os.path.join(SILVER_CSV_DIR, 'silver_test.csv')
lf.head(50).collect().write_csv(filepath)